## NVIDIA Nemotron Submission Demo Blackwell Compatibility Fixes

Proposed by **EdyVision**

The Kaggle utility script ships pre-compiled CUDA binaries targeting older GPU architectures. Blackwell GPU required three patches to train Nemotron successfully:

- Triton ptxas-blackwell fix — the binary in the read-only utility script lacks execute permissions. We copy it to a writable location, chmod +x, and patch Triton's compiler to use the fixed copy.
- Pure PyTorch causal_conv1d replacement — the pre-compiled causal_conv1d_cuda.so has no kernel image for Blackwell. We replace it with an equivalent pure-PyTorch implementation using F.conv1d with left-padding.
- Disable CUDA/Triton fast path — the Mamba mixer's fast path calls into the broken CUDA kernels. Disabling it forces the model to use a pure-PyTorch code path that works on any GPU architecture.

All patches are temporary. Remove them once Kaggle updates the utility script with Blackwell-compatible binaries.

**Cell Order Matters**

Run this notebook in order. Training will be slower than native CUDA due to the pure-PyTorch fallbacks, but this unblocks Blackwell users who are currently unable to train at all.

<em>Note: You will still need to proceed with your training logic, but hopefully this unblocks those attempting to use the Blackwell while the official patches come in. Happy Hacking!</em> 

## TEMPORARY PATCHING

### Copy utility and update perms

In [ ]:
import os, subprocess

DST = "/kaggle/working/nvidia-utility-script"
SRC = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"

os.makedirs(f"{DST}/triton/backends/nvidia/bin", exist_ok=True)
subprocess.run([
    "cp",
    f"{SRC}/triton/backends/nvidia/bin/ptxas-blackwell",
    f"{DST}/triton/backends/nvidia/bin/ptxas-blackwell"
], check=True)
subprocess.run(["chmod", "+x", f"{DST}/triton/backends/nvidia/bin/ptxas-blackwell"], check=True)

print("Copied and fixed ptxas-blackwell")

### Dependencies (yours may differ)

In [ ]:
# Dependency Imports
import torch
import os, subprocess, sys
import pandas as pd
import numpy as np
import kagglehub
import mamba_ssm
import re
import math
import subprocess

from typing import Optional
from datasets import Dataset
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, TaskType
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

### Triton Knobs Patch

In [ ]:
import triton.knobs as knobs
import triton.backends.nvidia.compiler as nv_compiler

DST = "/kaggle/working/nvidia-utility-script"
fixed_path = f"{DST}/triton/backends/nvidia/bin/ptxas-blackwell"

# Create a tool object that matches what Triton expects
class FixedNvidiaTool:
    def __init__(self, path, version="12.8"):
        self.path = path
        self.version = version

fixed_tool = FixedNvidiaTool(fixed_path)

# Patch the function that returns ptxas
_original_get_ptxas = nv_compiler.get_ptxas

def patched_get_ptxas(arch):
    if arch >= 100:
        return fixed_tool
    return _original_get_ptxas(arch)

nv_compiler.get_ptxas = patched_get_ptxas

# Also patch get_ptxas_version to use our fixed binary
def patched_get_ptxas_version(arch):
    import subprocess
    tool = patched_get_ptxas(arch)
    version = subprocess.check_output([tool.path, "--version"]).decode("utf-8")
    return version

nv_compiler.get_ptxas_version = patched_get_ptxas_version

print(f"Patched Triton to use {fixed_path}")
print(f"   Executable: {os.access(fixed_path, os.X_OK)}")

In [ ]:
# Disable the CUDA/Triton fast path in the model code
for mod_name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in mod_name and hasattr(mod, 'is_fast_path_available'):
        mod.is_fast_path_available = False
        print(f"Disabled fast path in {mod_name}")

In [ ]:
# ---------------------------------------------------------------
# Blackwell compatibility fix
# The pre-packaged causal_conv1d_cuda.so was compiled for older
# GPU architectures. This replaces the CUDA kernel calls with
# equivalent pure-PyTorch ops (F.conv1d with left-padding).
# The is_fast_path_available flag is also disabled to avoid
# selective_scan_cuda and Triton SSD kernels that similarly
# lack Blackwell support.
# ---------------------------------------------------------------

def causal_conv1d_fn_pure(x, weight, bias=None, seq_idx=None, initial_states=None,
                          return_final_states=False, final_states_out=None, activation=None):
    dtype_in = x.dtype
    x = x.to(weight.dtype)
    seqlen = x.shape[-1]
    dim, width = weight.shape
    if initial_states is None:
        out = F.conv1d(x, weight.unsqueeze(1), bias, padding=width - 1, groups=dim)
    else:
        x = torch.cat([initial_states, x], dim=-1)
        out = F.conv1d(x, weight.unsqueeze(1), bias, padding=0, groups=dim)
    out = out[..., :seqlen]
    if activation in ["silu", "swish"]:
        out = F.silu(out)
    out = out.to(dtype=dtype_in)
    if return_final_states:
        return out, final_states_out
    return out

import causal_conv1d
import causal_conv1d.causal_conv1d_interface as cci
causal_conv1d.causal_conv1d_fn = causal_conv1d_fn_pure
cci.causal_conv1d_fn = causal_conv1d_fn_pure

print("Patched causal_conv1d")

## Preview and Prep Data

In [ ]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

## Move onto your adapter work

In [ ]:
import os

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")

# Initialize LoRA Adapter (Rank 1)
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# YOUR CODE HERE
# --------------
# model.train() 
# --------------


# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

## Save Your work
Do not forget to zip only your adapter files (remember we copied the utility)

May need to do something like this:

```python
subprocess.run(
    f"cd {OUTPUT_DIR} && zip submission.zip adapter_config.json adapter_model.safetensors",
    shell=True, check=True
)
```

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')